# Part I: Basics

# Introduction

> We should be suspicious of any dataset (large or small) which appears perfect.
>
> --- David J. Hand

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.datasets import get_rdataset
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

# Load the same built-in R dataset through statsmodels
airquality = get_rdataset("airquality", "datasets").data
airquality.head()

## The problem of missing data

### Current practice

The mean of the numbers 1, 2 and 4 can be calculated in `Python` as

In [ ]:
 
y = np.array([1, 2, 4])
np.mean(y)
 

where `y` is an array containing three numbers, and where `np.mean(y)` is the `Python` expression that returns their mean. Now suppose that the last number is missing. `Python` and pandas commonly represent missing numeric values with `NaN`, which stands for “not a number”:

In [ ]:
 
# Retype the array y with np.nan instead of 4 in the last position. Recalculate the mean. What happens?
y1 = np.array([1, 2, np.nan])
np.mean(y1)
 

The mean is now undefined, and `Python` informs us about this outcome by setting the mean to `nan`. It is possible to use `np.nanmean()` to remove any missing data before calculating the mean:

In [ ]:
 
# Use np.nanmean(). What happens?
np.nanmean(y1)
 

This makes it possible to calculate a result, but of course the set of observations on which the calculations are based has changed. This may cause problems in statistical inference and interpretation.

It gets worse with multivariate analysis. For example, let us try to predict daily ozone concentration (ppb) from wind speed (mph) using the built-in `airquality` dataset. We fit a linear regression model by fitting an ordinary least squares regression model to predict daily ozone levels, as follows:

In [ ]:
 
fit = smf.ols("Ozone ~ Wind", data=airquality).fit()
len(airquality)
fit.nobs
 

**Why are the number of rows in the data set greater than the number of observations used in the fit?**

Next, we wish to plot the predicted ozone levels against the observed data, so we use `predict()` to calculate the predicted values, and add these to the data to prepare for plotting.

In [ ]:
 
fit = smf.ols("Ozone ~ Wind", data=airquality).fit()

# This deliberately shows the same problem: predictions are only available
# for complete cases in the fitted model.
complete_air = airquality[["Ozone", "Wind"]].dropna()
predicted = fit.predict(complete_air)

print("Rows in airquality:", len(airquality))
print("Rows with predictions:", len(predicted))
 

Argg... that doesn’t work! The error message tells us that the two datasets have a different number of rows. The `airquality` data has 153 rows, whereas there are only 116 predicted values. The problem, of course, is that there are missing data. The `lm()` function dropped any incomplete rows in the data. We find the indices of the first six cases by

In [ ]:
 
fit = smf.ols("Ozone ~ Wind", data=airquality).fit()
missing_for_model = airquality[["Ozone", "Wind"]].isna().any(axis=1)
np.where(missing_for_model)[0][:6]
 

The total number of deleted cases is found as

In [ ]:
 
fit = smf.ols("Ozone ~ Wind", data=airquality).fit()
missing_for_model = airquality[["Ozone", "Wind"]].isna().any(axis=1)
print(f"{missing_for_model.sum()} observations deleted due to missingness")
 

The number of missing values per variable in the data is

In [ ]:
 
fit = smf.ols("Ozone ~ Wind", data=airquality).fit()
airquality.isna().sum()
 

so in our regression model, all 37 deleted cases have missing ozone scores.

Removing the incomplete cases prior to analysis is known as *listwise deletion* or *complete-case analysis*. In `Python`, pandas provides methods such as `dropna()` and `isna()` to identify or remove incomplete cases.

The figure below plots the predicted against the observed values. Here we adopt the Abayomi convention for the colors [@ABAYOMI2008]: Blue refers to the observed part of the data, red to the synthetic part of the data (also called the *imputed values* or *imputations*), and black to the combined data (also called the *imputed data* or *completed data*).

In [ ]:
 
complete_air = airquality[["Ozone", "Wind"]].dropna().copy()
complete_air["predicted"] = fit.predict(complete_air)
airquality2 = complete_air
airquality2.head()
 

In [ ]:
 
fig, ax = plt.subplots(figsize=(7, 4))

ax.set_xlim(-20, 165)
ax.set_ylim(-20, 165)
ax.set_xlabel("Ozone measured (ppb)")
ax.set_ylabel("Ozone predicted (ppb)")
ax.axline((0, 0), slope=1, linestyle=":", linewidth=0.8, color="black")

# Observed complete cases
ax.scatter(airquality2["Ozone"], airquality2["predicted"], color="blue", label="Observed")

# Predicted values for missing Ozone cases with observed Wind
fit2 = smf.ols("Ozone ~ Wind", data=airquality).fit()
missing_ozone = airquality["Ozone"].isna() & airquality["Wind"].notna()
pred2 = fit2.predict(airquality.loc[missing_ozone, ["Wind"]])

for yhat in pred2:
    ax.axhline(yhat, color="red", linewidth=0.6, alpha=0.75)

ax.legend()
plt.show()
 

(ref:plotair9) Predicted versus measured ozone levels for the observed ( blue circles) and missing values (red lines).

The blue points are all from the complete cases, and the points for the incomplete cases are represented by red lines. Since there are no measured ozone levels in that part of the data, the possible values are indicated by 37 horizontal lines.

### What is the lesson of the figure?

Van Buuren is illustrating a fundamental problem with **regression imputation**. The blue points show the relationship between measured Ozone and predicted ozone for the observed cases. The red horizontal lines show that for the missing cases there is not one obvious correct value. Many ozone values are compatible with the same prediction. The uncertainty is large. Yet regression imputation later replaces each missing value with a **single point prediction**. The next sections criticize **deterministic regression imputation** because it treats uncertain predictions as though they were known exactly.

The dashed line is the 45-degree line ($y = x$). It is simply a visual reference:

$$\text{Predicted Ozone} = \text{Measured Ozone}$$

Points on the dashed line are predicted perfectly. The farther a point is from the line, the larger the prediction error. The figure illustrates an important limitation of regression imputation: a model may provide a reasonable prediction, but there is still substantial uncertainty about the true value. Replacing a missing value with a single predicted value ignores that uncertainty.

Listwise deletion allows the calculations to proceed, but it may introduce additional complexities in interpretation. Let’s try to find a better predictive model by including solar radiation (`Solar.R`) into the model as

In [ ]:
 
fit2 = smf.ols('Ozone ~ Wind + Q("Solar.R")', data=airquality).fit()
missing_for_model2 = airquality[["Ozone", "Wind", "Solar.R"]].isna().any(axis=1)
print(f"{missing_for_model2.sum()} observations deleted due to missingness")
 

Observe that the number of deleted days increased is now 42 since some rows had no value for `Solar.R`. Thus, changing the model altered the sample.

There are methodological and statistical issues associated with this procedure. Some questions that come to mind are:

- Can we compare the regression coefficients from both models?

- Should we attribute differences in the coefficients to changes in the model or to changes in the subsample?

- Do the estimated coefficients generalize to the study population?

- Do we have enough cases to detect the effect of interest?

- Are we making the best use of the costly collected data?

Getting the software to run is one thing, but this alone does not address the challenges posed by the missing data. Unless the analyst, or the software vendor, provides some way to work around the missing values, the analysis cannot continue because calculations on missing values are not possible. There are many approaches to circumvent this problem. Each of these affects the end result in a different way. Some solutions are clearly better than others, and there is no solution that will always work. This chapter reviews the major approaches, and discusses their advantages and limitations.

### Changing perspective on missing data

The standard approach to missing data is to delete them. It is illustrative to search for missing values in published data. @HAND1994 published a highly useful collection of small datasets across the statistical literature. The collection covers an impressive variety of topics. Only 13 out of the 510 datasets in the collection actually had a code for the missing data. In many cases, the missing data problem has probably been \`\`solved'' in some way, usually without telling us how many missing values there were originally. It is impossible to track down the original data for most datasets in Hand’s book. However, we can easily do this for dataset number 357, a list of scores of 34 athletes in 10 sport events at the 1988 Olympic decathlon in Seoul. The table itself is complete, but a quick search on the Internet revealed that initially 39 instead of 34 athletes participated. Five of them did not finish for various reasons, including the dramatic disqualification of the German favorite Jürgen Hingsen because of three false starts in the 100-meter sprint. It is probably fair to assume that deletion occurred silently in many of the other datasets.

The inclination to delete the missing data is understandable. Apart from the technical difficulties imposed by the missing data, the occurrence of missing data has long been considered a sign of sloppy research. It is all too easy for a referee to write:

> This study is weak because of the large amount of missing data.

Publication chances are likely to improve if there is no hint of missingness. @ORCHARD1972 [p. 697] remarked:

> Obviously the best way to treat missing data is not to have them.

Though there is a lot of truth in this statement, Orchard and Woodbury realized the impossibility of attaining this ideal in practice. The prevailing scientific practice is to downplay the missing data. Reviews on reporting practices are available in various fields: clinical trials [@WOOD2004; @POWNEY2014; @DIAZ2014; @AKL2015], cancer research [@BURTON2004], educational research [@PEUGH2004], epidemiology [@KLEBANOFF2008; @KARAHALIOS2012], developmental psychology [@JELICIC2009], general medicine [@MACKINNON2010], developmental pediatrics [@AYLWARD2010], and otorhinolaryngology, head and neck surgery [@NETTEN2017]. The picture that emerges from these studies is quite consistent:

- The presence of missing data is often not explicitly stated in the text;

- Default methods, such as listwise deletion are used without mentioning them;

- Different tables are based on different sample sizes;

- Model-based missing data methods, such as direct likelihood, full information maximum likelihood and multiple imputation, are notably underutilized.

Helpful resources include the STROBE [@VANDENBROUCKE2007] and CONSORT checklists and flow charts [@SCHULZ2010]. @GOMES2016 showed cases where the subset of full patient-reported outcomes is selected, leading to misleading results. @PALMER2017 suggested a classification scheme for the reasons of nonresponse in patient-reported outcomes.

Missing data are there, whether we like it or not. In the social sciences, it is nearly inevitable that some respondents will refuse to participate or to answer certain questions. In medical studies, attrition of patients is very common. The theory, methodology and software for handling incomplete data problems have been vastly expanded and refined over the last decades. The major statistical analysis packages now have facilities for performing the appropriate analyses. This book aims to contribute to a better understanding of the issues involved, and provides a methodology for dealing with incomplete data problems in practice.

## Concepts of MCAR, MAR and MNAR

Before we review a number of simple fixes for the missing data in Section \@ref(sec:simplesolutions) let us take a short look at the terms MCAR, MAR and MNAR. A more detailed definition of these concepts will be given later in Section \@ref(sec:notation). @RUBIN1976 classified missing data problems into three categories. In his theory every data point has some likelihood of being missing. The process that governs these probabilities is called the *missing data mechanism* or *response mechanism*. The model for the process is called the *missing data model* or *response model*.

If the probability of being missing is the same for all cases, then the data are said to be missing completely at random (MCAR). This effectively implies that causes of the missing data are unrelated to the data. We may consequently ignore many of the complexities that arise because data are missing, apart from the obvious loss of information. An example of MCAR is a weighing scale that ran out of batteries. Some of the data will be missing simply because of bad luck. Another example is when we take a random sample of a population, where each member has the same chance of being included in the sample. The (unobserved) data of members in the population that were not included in the sample are MCAR. While convenient, MCAR is often unrealistic for the data at hand.

If the probability of being missing is the same only within groups defined by the *observed* data, then the data are missing at random (MAR). MAR is a much broader class than MCAR. For example, when placed on a soft surface, a weighing scale may produce more missing values than when placed on a hard surface. Such data are thus not MCAR. If, however, we know surface type and if we can assume MCAR *within* the type of surface, then the data are MAR. Another example of MAR is when we take a sample from a population, where the probability to be included depends on some known property. MAR is more general and more realistic than MCAR. Modern missing data methods generally start from the MAR assumption.

If neither MCAR nor MAR holds, then we speak of missing not at random (MNAR). In the literature one can also find the term NMAR (not missing at random) for the same concept. MNAR means that the probability of being missing varies for reasons that are unknown to us. For example, the weighing scale mechanism may wear out over time, producing more missing data as time progresses, but we may fail to note this. If the heavier objects are measured later in time, then we obtain a distribution of the measurements that will be distorted. MNAR includes the possibility that the scale produces more missing values for the heavier objects (as above), a situation that might be difficult to recognize and handle. An example of MNAR in public opinion research occurs if those with weaker opinions respond less often. MNAR is the most complex case. Strategies to handle MNAR are to find more data about the causes for the missingness, or to perform what-if analyses to see how sensitive the results are under various scenarios.

Rubin’s distinction is important for understanding why some methods will work, and others not. His theory lays down the conditions under which a missing data method can provide valid statistical inferences. Most simple fixes only work under the restrictive and often unrealistic MCAR assumption. If MCAR is implausible, such methods can provide biased estimates.

## Ad-hoc solutions

### Listwise deletion

Complete-case analysis (listwise deletion) is the default way of handling incomplete data in many statistical packages, including `SPSS`, `SAS` and `Stata`. In `Python`, `dropna()` can be used for the same purpose. The procedure eliminates all cases with one or more missing values on the analysis variables. An important advantage of complete-case analysis is convenience. If the data are MCAR, listwise deletion produces unbiased estimates of means, variances and regression weights. Under MCAR, listwise deletion produces standard errors and significance levels that are correct for the reduced subset of data, but that are often larger relative to all available data.

A disadvantage of listwise deletion is that it is potentially wasteful. It is not uncommon in real life applications that more than half of the original sample is lost, especially if the number of variables is large. @KING2001 estimated that the percentage of incomplete records in the political sciences exceeded 50% on average, with some studies having over 90% incomplete records. It will be clear that a smaller subsample could seriously degrade the ability to detect the effects of interest.

If the data are not MCAR, listwise deletion can severely bias estimates of means, regression coefficients and correlations. @LITTLE2002 [pp. 41–44] showed that the bias in the estimated mean increases with the difference between means of the observed and missing cases, and with the proportion of the missing data. @SCHAFER2002 reported an elegant simulation study that demonstrates the bias of listwise deletion under MAR and MNAR. However, complete-case analysis is not always bad. The implications of the missing data are different depending on where they occur (outcomes or predictors), and the parameter and model form of the complete-data model. In the context of regression analysis, listwise deletion possesses some unique properties that make it attractive in particular settings. There are cases in which listwise deletion can provide better estimates than even the most sophisticated procedures. Since their discussion requires a bit more background than can be given here, we defer the treatment to Section \@ref(sec:when).

Listwise deletion can introduce inconsistencies in reporting. Since listwise deletion is automatically applied to the active set of variables, different analyses on the same data are often based on different subsamples. In principle, it is possible to produce one global subsample using all active variables. In practice, this is unattractive since the global subsample will always have fewer cases than each of the local subsamples, so it is common to create different subsets for different tables. It will be evident that this complicates their comparison and generalization to the study population.

In some cases, listwise deletion can lead to nonsensical subsamples. For example, the rows in the `airquality` dataset used in Section \@ref(sec:current) correspond to 154 consecutive days between May 1, 1973 and September 30, 1973. Deleting days affects the time basis. It would be much harder, if not impossible, to perform analyses that involve time, e.g., to identify weekly patterns or to fit autoregressive models that predict from previous days.

The opinions on the merits of listwise deletion vary. @MIETTINEN1985 [p. 231] described listwise deletion as

> …the only approach that assures that no bias is introduced under any circumstances…

a bold statement, but incorrect. At the other end of the spectrum we find @ENDERS2010 [p. 39]:

> In most situations, the disadvantages of listwise deletion far outweigh its advantages.

@SCHAFER2002 [p. 156] cover the middle ground:

> If a missing data problem can be resolved by discarding only a small part of the sample, then the method can be quite effective.

The leading authors in the field are, however, wary of providing advice about the percentage of missing cases below which it is still acceptable to do listwise deletion. @LITTLE2002 argue that it is difficult to formulate rules of thumb since the consequences of using listwise deletion depend on more than the missing data rate alone. @VACH1994 [p. 113] expressed his dislike for simplistic rules as follows:

> It is often supposed that there exists something like a critical missing rate up to which missing values are not too dangerous. The belief in such a global missing rate is rather stupid.

### Pairwise deletion

*Pairwise deletion*, also known as *available-case analysis*, attempts to remedy the data loss problem of listwise deletion. The method calculates the means and (co)variances on all observed data. Thus, the mean of variable $X$ is based on all cases with observed data on $X$, the mean of variable $Y$ uses all cases with observed $Y$-values, and so on. For the correlation and covariance, all data are taken on which both $X$ and $Y$ have non-missing scores. Subsequently, the matrix of summary statistics are fed into a program for regression analysis, factor analysis or other modeling procedures.

`SPSS`, `SAS` and `Stata` contain many procedures with an option for pairwise deletion. In `R` we can calculate the means and correlations of the `airquality` data under pairwise deletion in `R` as:

In [ ]:
 
data = airquality[["Ozone", "Solar.R", "Wind"]]

cov_pair = data.cov(min_periods=1)
cov_complete = data.dropna().cov()

print("Pairwise covariance")
display(cov_pair)
print("Complete-case covariance")
display(cov_complete)

np.linalg.eigvals(cov_pair)
 

The method is simple, and appears to use all available information. Under MCAR, it produces consistent estimates of mean, correlations and covariances [@LITTLE2002 p. 55]. The method has also some shortcomings. First, the estimates can be biased if the data are not MCAR. Further, the covariance and/or correlation matrix may not be positive definite, which is requirement for most multivariate procedures. We can check this by checking the Eigen values to see if they are all positive (which they are in this case, but that won't always be true). Problems are generally more severe for highly correlated variables [@LITTLE1992]. It is not clear what sample size should be used for calculating standard errors. Taking the average sample size yields standard errors that are too small[@LITTLE1992]. Also, pairwise deletion requires numerical data that follow an approximate normal distribution, whereas in practice we often have variables of mixed types.

The idea to use all available information is good, but the proper analysis of the pairwise matrix requires sophisticated optimization techniques and special formulas to calculate the standard errors [@VANPRAAG1985; @MARSH1998], which somewhat defeats its utility. Pairwise deletion works best used if the data approximate a multivariate normal distribution, if the correlations between the variables are low, and if the assumption of MCAR is plausible. It is not recommended for other cases.

For the airquality data, pairwise deletion seems OK. The covariance matrix looks reasonable (all positive Eigen values). The problem is that we have no guarantee that it will remain reasonable in other datasets.

However, even with a reasonable covariance matrix, there are still issues.

In [ ]:
 
vars_ = ["Ozone", "Solar.R", "Wind"]

n_pair = pd.DataFrame(index=vars_, columns=vars_, dtype=int)

for x in vars_:
    for y in vars_:
        n_pair.loc[x, y] = airquality[[x, y]].dropna().shape[0]

n_pair
 

The real conceptual issue with pairwise deletion is that the covariance matrix is being assembled from several different samples. Pairwise deletion uses different subsets of observations to estimate different means, variances, and covariances. Under what assumption can we believe that all of those subsets are representative of the same population?

### Mean imputation

A quick fix for the missing data is to replace them by the mean. We may use the mode for categorical data. Suppose we want to impute the mean in `Ozone` and `Solar.R` of the `airquality` data. `SPSS`, `SAS` and `Stata` have pre-built functions that substitute the mean. This book uses the `R` package `mice` [@VANBUUREN2011B]. This software is a contributed package that extends the functionality of `R`. Before `mice` can be used, it must be installed. An easy way to do this is to type:

In [ ]:
 
# If needed, install packages in your environment.
# Most scientific Python environments already include pandas, scikit-learn,
# statsmodels, numpy, and matplotlib.
 

In [ ]:
 
# Multiple imputation examples below use scikit-learn's IterativeImputer
# and statsmodels for regression modeling.
 

Imputing the mean in each variable can now be done by

In [ ]:
 
# Mean imputation
imp_mean = SimpleImputer(strategy="mean")
mean_completed_array = imp_mean.fit_transform(airquality)
mean_completed = pd.DataFrame(mean_completed_array, columns=airquality.columns)
 

The argument `method = mean` specifies mean imputation, the argument `m = 1` requests a single imputed dataset, and `maxit = 1` sets the number of iterations to 1 (no iteration). The latter two options can be left to their defaults with essentially the same result.

In [ ]:
 
data = mean_completed.copy()

plot_data = data.copy()
plot_data["status"] = np.where(airquality["Ozone"].isna(), "Imputed", "Observed")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for status, group in plot_data.groupby("status"):
    axes[0].hist(group["Ozone"], bins=range(0, 180, 10), alpha=0.45, label=status)
axes[0].set_xlabel("Ozone (ppb)")
axes[0].set_ylabel("Count")
axes[0].legend()

for status, group in plot_data.groupby("status"):
    axes[1].scatter(group["Solar.R"], group["Ozone"], alpha=0.85, label=status)
axes[1].set_ylim(-10, 170)
axes[1].set_xlabel("Solar Radiation (lang)")
axes[1].set_ylabel("Ozone (ppb)")
axes[1].legend()

plt.show()
 

(@ref:plotmeanimp) Mean imputation of `Ozone`. Blue indicates the observed data, red indicates the imputed values.

In [ ]:
 
# Calculate the mean and standard deviation of the complete data and the imputed data. What has happened?

Yobs = airquality["Ozone"]
Ycomp = data["Ozone"]

observed_idx = Yobs.notna()
imputed_idx = Yobs.isna()

summary_table = pd.DataFrame({
    "Group": ["Observed", "Imputed only", "Completed"],
    "n": [observed_idx.sum(), imputed_idx.sum(), len(Ycomp)],
    "Mean": [Yobs[observed_idx].mean(), Ycomp[imputed_idx].mean(), Ycomp.mean()],
    "SD": [Yobs[observed_idx].std(), Ycomp[imputed_idx].std(), Ycomp.std()]
})

summary_table
 

Mean imputation distorts the distribution in several ways. The figure below displays the distribution of `Ozone` after imputation. In the figure on the left, the red bar at the mean stands out. Imputing the mean here actually creates a bimodal distribution. The standard deviation in the imputed data is equal to 28.7, much smaller than from the observed data alone, which is 33. The figure on the right-hand side shows that the relation between `Ozone` and `Solar.R` is distorted because of the imputations. The correlation drops from 0.35 in the blue points to 0.3 in the combined data.

Mean imputation is a fast and simple fix for the missing data. However, it will underestimate the variance, disturb the relations between variables, bias almost any estimate other than the mean and bias the estimate of the mean when data are not MCAR. Mean imputation should perhaps only be used as a rapid fix when a handful of values are missing, and it should be avoided in general.

### Regression imputation

Regression imputation incorporates knowledge of other variables with the idea of producing smarter imputations. The first step involves building a model from the observed data. Predictions for the incomplete cases are then calculated under the fitted model, and serve as replacements for the missing data.

Suppose that we predict `Ozone` by linear regression from `Solar.R`.

In [ ]:
 
fit = smf.ols('Ozone ~ Q("Solar.R")', data=airquality).fit()

missing_ozone = airquality["Ozone"].isna() & airquality["Solar.R"].notna()

pred = fit.predict(airquality.loc[missing_ozone])

Yimp = airquality["Ozone"].copy()
Yimp.loc[missing_ozone] = pred
 

Now plot the imputed values versus the rest of the data.

In [ ]:
 
plot_data = pd.DataFrame({
    "Ozone": Yimp,
    "Solar.R": airquality["Solar.R"],
    "status": np.where(airquality["Ozone"].isna(), "Imputed", "Observed")
})

fig, ax = plt.subplots(figsize=(6, 4))
for status, group in plot_data.groupby("status"):
    ax.scatter(group["Solar.R"], group["Ozone"], alpha=0.85, label=status)

x_vals = np.linspace(plot_data["Solar.R"].min(), plot_data["Solar.R"].max(), 100)
y_vals = fit.params["Intercept"] + fit.params['Q("Solar.R")'] * x_vals
ax.plot(x_vals, y_vals, linestyle="--", color="black")

ax.set_xlabel("Solar Radiation (lang)")
ax.set_ylabel("Ozone (ppb)")
ax.legend()
plt.show()
 

(ref:plotregimp) Regression imputation: Imputing `Ozone` from the regression line.

In [ ]:
 
fit = smf.ols('Ozone ~ Q("Solar.R")', data=airquality).fit()

missing_rows = airquality["Ozone"].isna() & airquality["Solar.R"].notna()
pred = fit.predict(airquality.loc[missing_rows])

Yobs = airquality["Ozone"]
Yimp = Yobs.copy()
Yimp.loc[missing_rows] = pred

Yobs_only = Yobs.dropna()
Yimp_only = pred
Ycomp = Yimp

pd.DataFrame({
    "Group": ["Observed", "Imputed only", "Completed"],
    "Mean": [Yobs_only.mean(), Yimp_only.mean(), Ycomp.mean(skipna=True)],
    "SD": [Yobs_only.std(), Yimp_only.std(), Ycomp.std(skipna=True)]
})
 

### Another way to perform regression imputation

Another possibility for regression imputation uses `mice`:

In [ ]:
 
# Deterministic regression imputation can be approximated by fitting a regression
# and replacing missing values with predictions, as shown above.
 

The figure below shows the result. The imputed values correspond to the most likely values under the model. However, the ensemble of imputed values vary less than the observed values. It may be that each of the individual points is the best under the model, but it is very unlikely that the real (but unobserved) values of `Ozone` would have had this distribution. Imputing predicted values also has an effect on the correlation. The red points have a correlation of 1 since they are located on a line. If the red and blue dots are combined, then the correlation increases from 0.35 to 0.39. Note that this upward bias grows with the percent missing ozone levels (here 24%).

Regression imputation yields unbiased estimates of the means under MCAR, just like mean imputation, and of the regression weights of the imputation model if the explanatory variables are complete. Moreover, the regression weights are unbiased under MAR if the factors that influence the missingness are part of the regression model. In the example this corresponds to the situation where `Solar.R` would explain any differences in the probability that `Ozone` is missing. On the other hand, correlations are biased upwards, and the variability of the imputed data is systematically underestimated. The degree of underestimation depends on the explained variance and on the proportion of missing cases [@LITTLE2002 p. 64].

Regression imputation of predicted values can yield realistic imputations if the prediction is close to perfection. If so, the method reconstructs the missing parts from the available data. In essence, there was not really any information missing in the first place, it was only coded in a different form.

Regression imputation, as well as its modern incarnations in machine learning is probably the most dangerous of all methods described here. We may be led to believe that we’re to do a good job by preserving the relations between the variables. In reality however, regression imputation artificially strengthens the relations in the data. Correlations are biased upwards. Variability is underestimated. Imputations are too good to be true. Regression imputation is a recipe for false positive and spurious relations.

### Stochastic regression imputation

Stochastic regression imputation is a refinement of regression imputation attempts to address correlation bias by adding noise to the predictions. The following code imputes `Ozone` from `Solar.R` by stochastic regression imputation.

In [ ]:
 
# Stochastic regression imputation: prediction plus random residual noise
rng = np.random.default_rng(1)

data = airquality[["Ozone", "Solar.R"]].copy()
fit = smf.ols('Ozone ~ Q("Solar.R")', data=data).fit()

missing_rows = data["Ozone"].isna() & data["Solar.R"].notna()
pred = fit.predict(data.loc[missing_rows])
resid_sd = np.sqrt(fit.scale)

stochastic_completed = data.copy()
stochastic_completed.loc[missing_rows, "Ozone"] = pred + rng.normal(0, resid_sd, size=len(pred))
 

In [ ]:
 
data = stochastic_completed.copy()
plot_data = data.copy()
plot_data["status"] = np.where(airquality["Ozone"].isna(), "Imputed", "Observed")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for status, group in plot_data.groupby("status"):
    axes[0].hist(group["Ozone"].dropna(), bins=range(-40, 180, 10), alpha=0.45, label=status)
axes[0].set_xlabel("Ozone (ppb)")
axes[0].set_ylabel("Count")
axes[0].legend()

for status, group in plot_data.groupby("status"):
    axes[1].scatter(group["Solar.R"], group["Ozone"], alpha=0.85, label=status)
axes[1].set_ylim(-10, 170)
axes[1].set_xlabel("Solar Radiation (lang)")
axes[1].set_ylabel("Ozone (ppb)")
axes[1].legend()

plt.show()
 

(ref:plotsri) Stochastic regression imputation of `Ozone`.

The `method = norm.nob` argument requests a plain, non-Bayesian, stochastic regression method. This method first estimates the intercept, slope and residual variance under the linear model, then calculates the predicted value for each missing value, and adds a random draw from the residual to the prediction. We will come back to the details in Section \@ref(sec:linearnormal). The `seed` argument makes the solution reproducible. The figure below shows that the addition of noise to the predictions opens up the distribution of the imputed values, as intended.

Deterministic regression imputation places all imputed values exactly on the regression line and therefore understates variability. Stochastic regression imputation adds random residual variation so that imputed values resemble real observations more closely.

Note that some new complexities arise. There is one imputation with a negative value. Such values need not be due to the draws from the residual distribution, but can also be a consequence of the use of a linear model for non-negative data. In fact, The figure below shows several negative predicted values in the observed data. Since negative `Ozone` concentrations do not exist in the real world, we cannot consider negative values as plausible imputations. Note also that the high end of the distribution is not well covered. The observed data form a cone, i.e., the data are heteroscedastic, but the imputation model assumes equal dispersion around the regression line. The variability of `Ozone` increases up to the solar radiation level of 250 langleys, and decreases after that. Though it is unclear whether this is a genuine meteorological phenomenon, the imputation model does not account for this feature.

Nevertheless, stochastic regression imputation represents a major conceptual advance. Some analysts may find it counterintuitive to \`\`spoil'' the best prediction by adding random noise, yet this is precisely what makes it suitable for imputation. (Why?) A well-executed stochastic regression imputation preserves not only the regression weights, but also the correlation between variables. The main idea to draw from the residuals is very powerful, and forms the basis of more advanced imputation techniques.

### LOCF and BOCF

Last observation carried forward (LOCF) and baseline observation carried forward (BOCF) are ad-hoc imputation methods for longitudinal data. The idea is to take the previous observed value as a replacement for the missing data. When multiple values are missing in succession, the method searches for the last observed value.

The function `fill()` from the `tidyr` package applies LOCF by filling in the last known value. This is useful in situations where values are recorded only as they change, as in time-to-event data. For example, we may use LOCF to fill in `Ozone` by

In [ ]:
 
airquality2 = airquality.ffill()
 

In [ ]:
 
Oz = airquality["Ozone"]
Ozi = airquality2["Ozone"]

plot_data = pd.DataFrame({
    "Day": np.arange(1, len(Ozi) + 1),
    "Ozone": Ozi,
    "Status": np.where(Oz.isna(), "Carried Forward", "Observed")
}).query("Day <= 80")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(plot_data["Day"], plot_data["Ozone"], color="gray", linewidth=0.8)
for status, group in plot_data.groupby("Status"):
    ax.scatter(group["Day"], group["Ozone"], label=status)
ax.set_xlabel("Day number")
ax.set_ylabel("Ozone (ppb)")
ax.set_title("Last Observation Carried Forward")
ax.legend()
plt.show()
 

(ref:locf2) Imputation of `Ozone` by last observation carried forward (LOCF).

The figure below plots the results of the first 80 days of the `Ozone` series. The stretches of red dots indicate the imputations, and are constant within the same batch of missing ozone levels. The real, unseen values are likely to vary within these batches, so applying LOCF here gives implausible imputations.

LOCF is convenient because it generates a complete dataset. It can be applied with confidence in cases where we are certain what the missing values should be, for example, for administrative variables in longitudinal data. For outcomes, LOCF is dubious. The method has long been used in clinical trials. The U.S. Food and Drug Administration (FDA) has traditionally viewed LOCF as the preferred method of analysis, considering it conservative and less prone to selection than listwise deletion. However, @MOLENBERGHS2007 [pp. 47–50] show that the bias can operate in both directions, and that LOCF can yield biased estimates even under MCAR. LOCF needs to be followed by a proper statistical analysis method that distinguishes between the real and imputed data. This is typically not done however. Additional concerns about a reversal of the time direction are given in @KENWARD2009.

The Panel on Handling Missing Data in Clinical Trials recommends that LOCF and BOCF should not be used as the primary approach for handling missing data unless the assumptions that underlie them are scientifically justified [@NRC2010 p. 77].

### Indicator method

Suppose that we want to fit a regression, but there are missing values in one of the explanatory variables. The indicator method [@MIETTINEN1985 p. 232] replaces each missing value by a zero and extends the regression model by the response indicator. The procedure is applied to each incomplete variable. The user analyzes the extended model instead of the original.

In `R` the indicator method can be coded as

In [ ]:
 
mean_completed = pd.DataFrame(
    SimpleImputer(strategy="mean").fit_transform(airquality),
    columns=airquality.columns
)

airquality2 = mean_completed.copy()
airquality2["r_Ozone"] = airquality["Ozone"].isna()

fit = smf.ols("Wind ~ Ozone + r_Ozone", data=airquality2).fit()
fit.summary()
 

Observe that since the missing data are in `Ozone` we needed to reverse the direction of the regression model.

The indicator method has been popular in public health and epidemiology. An advantage is that the indicator method retains the full dataset. Also, it allows for systematic differences between the observed and the unobserved data by inclusion of the response indicator, and could be more efficient. @WHITE2005 pointed out that the method can be useful to estimate the treatment effect in randomized trials when a baseline covariate is partially observed. If the missing data are restricted to the covariate, if the interest is solely restricted to estimation of the treatment effect, if compliance to the allocated treatment is perfect and if the model is linear without interactions, then using the indicator method for that covariate yields an unbiased estimate of the treatment effect. This is true even if the missingness depends on the covariate itself. Additional work can be found in [@GROENWOLD2012; @SULLIVAN2016]. It is not yet clear whether the coverage of the confidence interval around the treatment estimate will be satisfactory for multiple incomplete baseline covariates.

The conditions under which the indicator method works may not be met in practice. For example, the method does not allow for missing data in the outcome, and generally fails in observational data. It has been shown that the method can yield severely biased regression estimates, even under MCAR and for low amounts of missing data [@VACH1991; @GREENLAND1995; @KNOL2010]. The indicator method may have its uses in particular situations, but fails as a generic method to handle missing data.

### Summary

|            |      |            |             |                |
|------------|------|:----------:|-------------|----------------|
|            |      |  Unbiased  |             | Standard Error |
|            | Mean | Reg Weight | Correlation |                |
| Listwise   | MCAR |    MCAR    | MCAR        | Too large      |
| Pairwise   | MCAR |    MCAR    | MCAR        | Complicated    |
| Mean       | MCAR |     –      | –           | Too small      |
| Regression | MAR  |    MAR     | –           | Too small      |
| Stochastic | MAR  |    MAR     | MAR         | Too small      |
| LOCF       | –    |     –      | –           | Too small      |
| Indicator  | –    |     –      | –           | Too small      |

: (#tab:simple) Overview of assumptions made by ad-hoc methods.

Table \@ref(tab:simple) provides a summary of the methods discussed in this section. The table addresses two topics: whether the method yields the correct results on average (unbiasedness), and whether it produces the correct standard error. Unbiasedness is evaluated with respect to three types of estimates: the mean, the regression weight (with the incomplete variable as dependent) and the correlation.

The table identifies the assumptions on the missing data mechanism each method must make in order to produce unbiased estimates. The first line of the table should be read as follows:

1.  *Listwise deletion* produces an unbiased estimate of the *mean* provided that the data are *MCAR*;

2.  *Listwise deletion* produces an estimate of the standard error that is *too large*.

The interpretation of the other lines is similar. The “–” sign in some cells indicates that the method cannot produce unbiased estimates. Observe that both deletion methods require MCAR for all types. Regression imputation and stochastic regression imputation can yield unbiased estimates under MAR. In order to work, the model needs to be correctly specified. LOCF and the indicator method are incapable of providing consistent estimates, even under MCAR. Note that some special cases are not covered in Table \@ref(tab:simple). For example, listwise deletion is unbiased under two special MNAR scenarios (cf. Section \@ref(sec:when)).

Listwise deletion produces standard errors that are correct for the subset of complete cases, but in general too large for the entire dataset. Calculation of standard errors under pairwise deletion is complicated. The standard errors after single imputation are too small since the standard calculations make no distinction between the observed data and the imputed data. Correction factors for some situations have been developed [@SCHAFER2000], but a more convenient solution is multiple imputation.

## Multiple imputation in a nutshell

### Procedure

Multiple imputation creates $m>1$ complete datasets. Each of these datasets is analyzed by standard analysis software. The $m$ results are pooled into a final point estimate plus standard error by pooling rules (“Rubin’s rules”). Figure \@ref(fig:miflow) illustrates the three main steps in multiple imputation: imputation, analysis and pooling.

In [ ]:
 
from IPython.display import Image, display
from pathlib import Path

image_path = Path("ch01-miflow-1.png")
if image_path.exists():
    display(Image(filename=str(image_path)))
else:
    print("Image file ch01-miflow-1.png not found in the current directory.")
 

(ref:miflow) Scheme of main steps in multiple imputation.

The analysis starts with observed, incomplete data. Multiple imputation creates several complete versions of the data by replacing the missing values by plausible data values. These plausible values are drawn from a distribution specifically modeled for each missing entry. Figure \@ref(fig:miflow) portrays $m=3$ imputed datasets. In practice, $m$ is often taken larger (cf. Section \@ref(sec:howmany)). The number $m=3$ is taken here just to make the point that the technique creates multiple versions of the imputed data. The three imputed datasets are identical for the observed data entries, but differ in the imputed values. The magnitude of these difference reflects our uncertainty about what value to impute.

The second step is to estimate the parameters of interest from each imputed dataset. This is typically done by applying the analytic method that we would have used had the data been complete. The results will differ because their input data differ. It is important to realize that these differences are caused only because of the uncertainty about what value to impute.

The last step is to pool the $m$ parameter estimates into one estimate, and to estimate its variance. The variance combines the conventional sampling variance (within-imputation variance) and the extra variance caused by the missing data extra variance caused by the missing data (between-imputation variance). Under the appropriate conditions, the pooled estimates are unbiased and have the correct statistical properties.

### Reasons to use multiple imputation

Multiple imputation [@RUBIN1987; @RUBIN1996] solves the problem of “too small” standard errors in Table \@ref(tab:simple). Multiple imputation is unique in the sense that it provides a mechanism for dealing with the inherent uncertainty of the imputations themselves.

Our level of confidence in a particular imputed value is expressed as the variation across the $m$ completed datasets. For example, in a disability survey, suppose that the respondent answered the item whether he could walk, but did not provide an answer to the item whether he could get up from a chair. If the person can walk, then it is highly likely that the person will also be able to get up from the chair. Thus, for persons who can walk, we can draw a “yes” for missing “getting up from a chair” with a high probability, say 0.99, and use the drawn value as the imputed value. In the extreme, if we are really certain, we always impute the same value for that person. In general however, we are less confident about the true value. Suppose that, in a growth study, height is missing for a subject. If we only know that this person is a woman, this provides some information about likely values, but not so much. So the range of plausible values from which we draw is much larger here. The imputations for this woman will thus vary a lot over the different datasets. Multiple imputation is able to deal with both high-confidence and low-confidence situations equally well.

Another reason to use multiple imputation is that it separates the solution of the missing data problem from the solution of the complete-data problem. The missing-data problem is solved first, the complete-data problem next. Though these phases are not completely independent, the answer to the scientifically interesting question is not obscured anymore by the missing data. The ability to separate the two phases simplifies statistical modeling, and hence contributes to a better insight into the phenomenon of scientific study.

### Example of multiple imputation

Continuing with the `airquality` dataset, it is straightforward to apply multiple imputation. The following code imputes the missing data twenty times, fits a linear regression model to predict `Ozone` in each of the imputed datasets, pools the twenty sets of estimated parameters, and calculates the Wald statistics for testing significance of the weights.

In [ ]:
 
# Multiple imputation using scikit-learn's IterativeImputer
# This is an analogous Python workflow, not a byte-for-byte replacement for R's mice.

features = ["Ozone", "Wind", "Temp", "Solar.R"]
imputations = []
models = []

for seed in range(1, 21):
    imputer = IterativeImputer(random_state=seed, sample_posterior=True, max_iter=20)
    completed_array = imputer.fit_transform(airquality[features])
    completed = pd.DataFrame(completed_array, columns=features)
    imputations.append(completed)
    models.append(smf.ols('Ozone ~ Wind + Temp + Q("Solar.R")', data=completed).fit())

# Pool coefficients using Rubin's rules
params = pd.DataFrame([m.params for m in models])
variances = pd.DataFrame([m.bse**2 for m in models])

pooled_params = params.mean()
within_var = variances.mean()
between_var = params.var(ddof=1)
m = len(models)
total_var = within_var + (1 + 1/m) * between_var
pooled_se = np.sqrt(total_var)

pooled_summary = pd.DataFrame({
    "coef": pooled_params,
    "std_err": pooled_se,
    "t": pooled_params / pooled_se
})

pooled_summary
 

There is much more to say about each of these steps, but it shows that multiple imputation need not be a daunting task. Assuming we have set `options(na.action = na.omit)`, fitting the same model to the complete cases can be done by

In [ ]:
 
fit = smf.ols('Ozone ~ Wind + Temp + Q("Solar.R")', data=airquality).fit()
fit.summary().tables[1]
 

The solutions are nearly identical here, which is due to the fact that most missing values occur in the outcome variable. The standard errors of the multiple imputation solution are slightly smaller than in the complete-case analysis. Multiple imputation is often more efficient than complete-case analysis. Depending on the data and the model at hand, the differences can be dramatic.

In [ ]:
 
# Use the first completed data set
data = imputations[0].copy()

plot_data = data.copy()
plot_data["status"] = np.where(airquality["Ozone"].isna(), "Imputed", "Observed")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for status, group in plot_data.groupby("status"):
    axes[0].hist(group["Ozone"], bins=range(0, 180, 10), alpha=0.45, label=status)
axes[0].set_xlabel("Ozone (ppb)")
axes[0].set_ylabel("Count")
axes[0].legend()

for status, group in plot_data.groupby("status"):
    axes[1].scatter(group["Solar.R"], group["Ozone"], alpha=0.85, label=status)
axes[1].set_ylim(-10, 170)
axes[1].set_xlabel("Solar Radiation (lang)")
axes[1].set_ylabel("Ozone (ppb)")
axes[1].legend()

plt.show()
 

(ref:plotairmi) Multiple imputation of `Ozone`. Plotted are the imputed values from the first imputation.

The figure below shows the distribution and scattergram for the observed and imputed data combined. The imputations are taken from the first completed dataset. The blue and red distributions are quite similar. Problems with the negative values as in The figure below are now gone since the imputation method used observed data as donors to fill the missing data. Section \@ref(sec:pmm) describes the method in detail. Note that the red points respect the heteroscedastic nature of the relation between `Ozone` and `Solar.R`. All in all, the red points look as if they could have been measured if they had not been missing. The reader can easily recalculate the solution and inspect these plots for the other imputations.

Your turn: Pick another completed data set (change complete(imp, 1) to complete(imp,m), where m = 2:20). What do you see?

In [ ]:
 
Oz = airquality["Ozone"]
colors = ["red", "blue", "darkgreen", "purple", "orange"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, len(Oz) + 1), Oz, color="black", linewidth=0.8, label="Observed")

missing_days = np.where(Oz.isna())[0] + 1

for i in range(5):
    completed = imputations[i]
    ax.scatter(
        missing_days,
        completed.loc[Oz.isna(), "Ozone"],
        color=colors[i],
        label=f"Imputation {i+1}",
        s=25
    )

ax.set_xlabel("Day number")
ax.set_ylabel("Ozone (ppb)")
ax.legend(loc="upper left", frameon=False)
plt.show()
 

(ref:air10) Multiple imputation of `Ozone`. Plotted are the observed values (in blue) and the multiply imputed values (in red).

The figure below plots the completed `Ozone` data. The imputed data of all five imputations are plotted for the days with missing `Ozone` scores. In order to avoid clutter, the lines that connect the dots are not drawn for the imputed values. Note that the pattern of imputed values varies substantially over the days. At the beginning of the series, the values are low and the spread is small, in particular for the cold and windy days 25–27. The small spread for days 25–27 indicates that the model is quite sure of these values. High imputed values are found around the hot and sunny days 35–42, whereas the imputations during the moderate days 52–61 are consistently in the moderate range. Note how the available information helps determine sensible imputed values that respect the relations between wind, temperature, sunshine and ozone.

One final point. The `airquality` data is a time series of 153 days. It is well known that the standard error of the ordinary least squares (OLS) estimate is inefficient (too large) if the residuals have positive serial correlation [@HARVEY1981]. The first three autocorrelations of the `Ozone` are indeed large: 0.48, 0.31 and 0.29. The residual autocorrelations are however small and within the confidence interval: 0.13, $-0.02$ and 0.04. The inefficiency of OLS is thus negligible here.

## Goal of the book

The main goal of this book is to add multiple imputation to the tool chest of practitioners. The text explains the ideas underlying multiple imputation, discusses when multiple imputation is useful, how to do it in practice and how to report the results of the steps taken.

The original computations in the source text are done with the help of the `R` package `mice`, written by Karin Groothuis-Oudshoorn and Stef van Buuren [@VANBUUREN2011B]. This Python adaptation uses pandas, statsmodels, and scikit-learn to demonstrate analogous ideas. Online materials that accompany the book can be found on www.multiple-imputation.com. My hope is that this hands-on approach will facilitate understanding of the key ideas in multiple imputation.

## What I didn't cover

The field of missing data research is vast. I focused on multiple imputation. I did not attempt cover the enormous body of literature on alternative approaches to incomplete data. This section briefly reviews three of these approaches.

### Prevention

With the exception of @MCKNIGHT2007 [Chapter 4], books on missing data do not mention prevention. Yet, prevention of the missing data is the most direct attack on problems caused by the missing data. Prevention is fully in spirit with the quote of Orchard and Woodbury given on p. . There is a lot one could do to prevent missing data. The remainder of this section lists point-wise advice.

Minimize the use of intrusive measures, like blood samples. Visit the subject at home. Use incentives to stimulate response, and try to match up the interviewer and respondent on age and ethnicity. Adapt the mode of the study (telephone, face to face, web questionnaire, and so on) to the study population. Use a multi-mode design for different groups in your study. Quickly follow-up for people that do not respond, and where possible try to retrieve any missing data from other sources.

In experimental studies, try to minimize the treatment burden and intensity where possible. Prepare a well-thought-out flyer that explains the purpose and usefulness of your study. Try to organize data collection through an authority, e.g., the patient’s own doctor. Conduct a pilot study to detect and smooth out any problems.

Economize on the number of variables collected. Only collect the information that is absolutely essential to your study. Use short forms of measurement instruments where possible. Eliminate vague or ambivalent questionnaire items. Use an attractive layout of the instruments. Refrain from using blocks of items that force the respondent to stay on a particular page for a long time. Use computerized adaptive testing where feasible. Do not allow other studies to piggy-back on your data collection efforts.

Do not overdo it. Many Internet questionnaires are annoying because they force the respondent to answer. Do not force your respondent. The result will be an apparently complete dataset with mediocre data. Respect the wish of your respondent to skip items. The end result will be more informative.

Use double coding in the data entry, and chase up any differences between the versions. Devise nonresponse forms in which you try to find out why people they did not respond, or why they dropped out.

Last but not least, consult experts. Many academic centers have departments that specialize in research methodology. Sound expert advice may turn out to be extremely valuable for keeping your missing data rate under control.

Most of this advice can be found in books on research methodology and data quality. Good books are @SHADISH2001, @DELEEUW2008, @DILLMAN2008 and @GROVES2009.

### Weighting procedures

Weighting is a method to reduce bias when the probability to be selected in the survey differs between respondents. In sample surveys, the responders are weighted by design weights, which are inversely proportional to their probability of being selected in the survey. If there are missing data, the complete cases are re-weighted according to design weights that are adjusted to counter any selection effects produced by nonresponse. The method is widely used in official statistics. Relevant pointers include @COCHRAN1977 and @SARNDAL1992 and @BETHLEHEM2002.

The method is relatively simple in that only one set of weights is needed for all incomplete variables. On the other hand, it discards data by listwise deletion, and it cannot handle partial response. Expressions for the variance of regression weights or correlations tend to be complex, or do not exist. The weights are estimated from the data, but are generally treated as fixed. The implications for this are unclear [@LITTLE2002 p. 53].

There has been interest recently in improved weighting procedures that are “double robust” [@SCHARFSTEIN1999; @BANG2005]. This estimation method requires specification of three models: Model A is the scientifically interesting model, Model B is the response model for the outcome, and model C is the joint model for the predictors and the outcome. The dual robustness property states that: if either Model B or Model C is wrong (but not both), the estimates under Model A are still consistent. This seems like a useful property, but the issue is not free of controversy [@KANG2007].

### Likelihood-based approaches

Likelihood-based approaches define a model for the observed data. Since the model is specialized to the observed values, there is no need to impute missing data or to discard incomplete cases. The inferences are based on the likelihood or posterior distribution under the posited model. The parameters are estimated by maximum likelihood, the EM algorithm, the sweep operator, Newton–Raphson, Bayesian simulation and variants thereof. These methods are smart ways to skip over the missing data, and are known as direct likelihood, full information maximum likelihood (FIML), and more recently, pairwise likelihood estimation.

Likelihood-based methods are, in some sense, the “royal way” to treat missing data problems. The estimated parameters nicely summarize the available information under the assumed models for the complete data and the missing data. The model assumptions can be displayed and evaluated, and in many cases it is possible to estimate the standard error of the estimates.

Multiple imputation extends likelihood-based methods by adding an extra step in which imputed data values are drawn. An advantage of this is that it is generally easier to calculate the standard errors for a wider range of parameters. Moreover, the imputed values created by multiple imputation can be inspected and analyzed, which helps us to gauge the effect of the model assumptions on the inferences.

The likelihood-based approach receives an excellent treatment in the book by @LITTLE2002. A less technical account that should appeal to social scientists can be found in @ENDERS2010 [chapters 3–5]. @MOLENBERGHS2007 provide a hands-on approach of likelihood-based methods geared toward clinical studies, including extensions to data that are MNAR. The pairwise likelihood method was introduced by @KATSIKATSOU2012 and has been implemented in `lavaan`.

## Structure of the book

The text, code, and examples here come from Stef van Buuren's online book: [Flexible Imputation of Missing Data](https://stefvanbuuren.name/fimd/). I used ChatGPT 5.5 from SMU's Business account to update the code.

This book consists of three main parts: basics, case studies and extensions. Chapter \@ref(ch:mi) reviews the history of multiple imputation and introduces the notation and theory. Chapter \@ref(ch:univariate) provides an overview of imputation methods for univariate missing data. Chapter \@ref(ch:multivariate) distinguishes three approaches to attack the problem of multivariate missing data. Chapter \@ref(ch:analysis) reviews issues pertaining to the analysis of the imputed datasets.

Chapter \@ref(ch:practice) discusses practical issues for multivariate missing data. Chapter \@ref(ch:multilevel) discusses the problem how to impute for nested data so as to preserve the multilevel structure. Chapter \@ref(ch:ice) explores the use of multiple imputation to estimate individual causal effects.

Chapters \@ref(ch:measurement)–\@ref(ch:longitudinal) contain case studies of the techniques described in the previous chapters. Chapter \@ref(ch:measurement) deals with “problems with the columns,” while Chapter \@ref(ch:selection) addresses “problems with the rows”. Chapter \@ref(ch:longitudinal) discusses studies on problems with both rows and columns.

Chapter \@ref(ch:conclusion) concludes the main text with a discussion of limitations and pitfalls, reporting guidelines, alternative applications and future extensions.

## Exercises

 {exercise, name = "Reporting practice", label = "report"}
What are the reporting practices in your field? Take a random sample 
of articles that have appeared during the last 10 years in the leading 
journal in your field. Select only those that present quantitative 
analyses, and address the following topics:
  
1. Did the authors report that there were missing data?
  
2. If not, can you infer from the text that there must have
   been missing data?
  
3. Did the authors discuss how they handled the missing
   data?
  
4. Were the missing data properly addressed?
  
5. Can you detect a trend over time in reporting practice?
  
6. Would the editors of the journal be interested in your findings?

 

 {exercise, name = "Loss of information", label = "caseslost"}
Suppose that a dataset consists of 100 cases and 10 variables. 
Each variable contains 10% missing values. What is the 
largest possible subsample under listwise deletion? What
is the smallest? If each variable is MCAR, how many cases will
remain?
 

 {exercise, name = "Stochastic regression imputation", label = "sriexr"}
The correlation of the data in The figure below is 
equal to 0.33. This is relatively low compared to the other 
correlations reported in Section \@ref(sec:simplesolutions). 
This seems to contradict the statement that stochastic regression 
imputation does not bias the correlation. Could this low correlation 
be due to random variation?

1. Rerun the code with a different `seed` value. What is the 
   correlation now?
     
2. Write a loop to apply apply stochastic regression imputation 
   with the `seed` increasing from 1 to 1000. Calculate the 
   regression weight and the correlation for each solution, and 
   plot the histogram. What are the mean, minimum and maximum 
   values of the correlation?
     
3. Do your results indicate that stochastic regression imputation 
   alters the correlation?

 

 {exercise, name = "Stochastic regression imputation (continued)", label = "sricontinued"}
The largest correlation found in the previous exercise exceeds the value found
in Section \@ref(sec:regimp). This seems odd since the correlation of
the imputed values under regression imputation is equal to 1, and hence the 
imputed data have a maximal contribution to the overall correlation.

1. Can you explain why this could happen?

2. Adapt the code from the previous exercise to test your explanation. 
   Was your explanation satisfactory?

3. If not, can you think of another reason, and test that?
   Hint: Find out what is special about the solutions with the
   largest correlations.

 

 {exercise, name = "Nonlinear model", label = "nonlinear"}
The model fitted to the `airquality` data in Section \@ref(sec:miexample) 
is a simple linear model. Inspection of the residuals reveals that 
there is a slight curvature in the average of the residuals.

1. Start from the completed cases, and use `plot(fit)` to obtain 
   diagnostic plots. Can you explain why the curvature shows up?

2. Experiment with solutions, e.g., by transforming `Ozone` or 
   by adding a quadratic term to the model. Can you make the 
   curvature disappear? Does the amount of explained variance 
   increase?

3. Does the curvature also show up in the imputed data? If so, 
   does the same solution work? Hint: You can assess the 
   $j^\mathrm{th}$ fitted model by `getfit(fit, j)`, where
   `fit` was created by `with(imp,...)`.

4. Advanced: Do you think your solution would necessitate
   drawing new imputations?

 

# References